# Introduction to LangChain

**Author:** Shinin Varongchayakul

**Date:** 30 Jan 2026

## 1. API Key

In [12]:
# Import packages
from pathlib import Path
from dotenv import load_dotenv
import os

# Get .env file path
PROJECT_ROOT = Path.cwd().parents[2]
env_path = PROJECT_ROOT / ".env"

# Load variables from .env
load_dotenv(env_path, override=True)

# Get Gemini API key
gemini_api_key = os.getenv("GEMINI_API_KEY_01")

## 2. LLM

In [13]:
# Import package
from langchain_google_genai import ChatGoogleGenerativeAI

# Create model instance
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.5,
    api_key=gemini_api_key
)

## 3. Prompt Template

In [ ]:
# Import package
from langchain_core.prompts import ChatPromptTemplate

# Define system prompt
system_prompt = """
You are an expert curator of mental models across science, philosophy, and applied reasoning.

Your task is to explain mental models clearly and accurately using a fixed schema.

If the origin of a model is unclear or debated, state that explicitly.

Do not invent historical sources. Be concise and concrete.
"""

# Define user prompt
user_prompt = "Explain the following mental model: {model_query}"

# Create prompt template
prompts = ChatPromptTemplate.from_messages(
    [
        # System prompt
        ("system", system_prompt.strip()),

        # User prompt
        ("human", user_prompt)
    ]
)

In [ ]:
# Inspect prompt template
prompts.format_messages(model_query="Pareto Principle")

[SystemMessage(content='You are an expert curator of mental models across science, philosophy, and applied reasoning.\n\nYour task is to explain mental models clearly and accurately using a fixed schema.\n\nIf the origin of a model is unclear or debated, state that explicitly.\n\nDo not invent historical sources. Be concise and concrete.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Explain the following mental model: Pareto Principle', additional_kwargs={}, response_metadata={})]

## 4. Output Format

In [ ]:
# Import packages
from pydantic import BaseModel, Field
from typing import List, Literal

# Define output structure
class MentalModel(BaseModel):

    # Mental model name
    model_name: str = Field(description="The commonly accepted name of the mental model")

    # Source/origin
    origin: str = Field(
        description="Where the model comes from (person, book, field, or cultural origin)"
    )

    # Brief description
    description: str = Field(
        description="A brief explanation of what the mental model is and why it matters"
    )

    # Example
    example: str = Field(
        description="A concrete real-world example illustrating the mental model"
    )

    # Tags
    tags: List[str] = Field(
        description="Short tags such as decision-making, systems thinking, learning, philosophy"
    )

In [17]:
# Add output structure to LLM
llm_with_structured_output = llm.with_structured_output(MentalModel)

## 5. Chain

In [ ]:
# Build chain
chain = prompts | llm_with_structured_output

## 6. Single Run

In [19]:
# Run query
result = chain.invoke({"model_query": "Compound Interest"})

In [20]:
# Import package
import json

# Load result
result_dict = result.model_dump()

# Print
print(json.dumps(result_dict, indent=4))

{
    "model_name": "Compound Interest",
    "origin": "Finance, Mathematics; concept dates back to ancient times, formalized in the Renaissance.",
    "description": "Compound interest is the interest on a loan or deposit calculated based on both the initial principal and the accumulated interest from previous periods. It is often called 'interest on interest' and leads to exponential growth over time, making it a powerful force in finance for both wealth creation and debt accumulation.",
    "example": "If you invest $1,000 at an annual interest rate of 5% compounded annually, after the first year you'll have $1,050. In the second year, the 5% interest is calculated on $1,050, not just the original $1,000, leading to a balance of $1,102.50. This snowball effect accelerates over decades, significantly increasing the total return compared to simple interest.",
    "tags": [
        "Finance",
        "Economics",
        "Wealth Building",
        "Decision-making",
        "Long-term 

## 7. Batch run

In [23]:
# Create list of mental models
mental_model_queries = [
    "First Principles Thinking",
    "Occam's Razor",
    "Confirmation Bias"
]

# Create batch inputs
batch_inputs = [{"model_query": query} for query in mental_model_queries]

# Run queries
results = chain.batch(batch_inputs)

# Instantiate collector
query_collector = [result.model_dump() for result in results]

In [24]:
# Instantiate counter
i = 1

# Loop through elements in collector
for result in query_collector:

    # Print result
    print(f"👉 Query {i}:")
    print(json.dumps(result, indent=4))
    print("\n")

    # Add 1 to counter
    i += 1

👉 Query 1:
{
    "model_name": "First Principles Thinking",
    "origin": "Often attributed to Aristotle; popularized in modern business by Elon Musk.",
    "description": "First Principles Thinking involves breaking down complex problems into their most basic, fundamental truths or 'first principles,' rather than reasoning by analogy or conventional wisdom. It matters because it allows for innovative solutions by challenging assumptions and building new knowledge from the ground up.",
    "example": "Instead of accepting the high cost of batteries for electric cars, Elon Musk famously broke down a battery into its constituent raw materials (cobalt, nickel, lithium, etc.) to understand their actual cost, then sought ways to procure and assemble them more efficiently, leading to significant cost reductions and innovation.",
    "tags": [
        "Problem-solving",
        "Innovation",
        "Critical Thinking",
        "Decision-making"
    ]
}


👉 Query 2:
{
    "model_name": "Occam